In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/energy_features_v3.parquet"
)

print(df.shape)

(9101, 31)


In [2]:
FEATURES = [
    "lag_1",
    "lag_7",
    "lag_30",

    "rolling_mean_7",
    "rolling_mean_30",

    "rolling_std_7",
    "rolling_std_30",

    "year",
    "month",
    "quarter",
    "day_of_week",
    "day_of_year",
    "is_weekend",

    "is_holiday",
    "is_christmas",
    "is_new_year",
    "is_easter",

    "temperature_mean",
    "temperature_max",
    "temperature_min",

    "rainfall",
    "wind_speed",

    "temp_lag_1",
    "rainfall_lag_1",
    "wind_lag_1",

    "EMBEDDED_SOLAR_GENERATION",
    "EMBEDDED_WIND_GENERATION",

    "solar_available",
    "wind_available"
]

TARGET = "ND"

In [3]:
split_date = "2024-01-01"

train = df[
    df["SETTLEMENT_DATE"] < split_date
]

test = df[
    df["SETTLEMENT_DATE"] >= split_date
]

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

print(X_train.shape)
print(X_test.shape)

(8370, 29)
(731, 29)


In [4]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [5]:
predictions = model.predict(
    X_test
)

In [6]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = (
    mean_squared_error(
        y_test,
        predictions
    ) ** 0.5
)

mape = (
    mean_absolute_percentage_error(
        y_test,
        predictions
    ) * 100
)

print("MAE :", round(mae,2))
print("RMSE:", round(rmse,2))
print("MAPE:", round(mape,2), "%")

MAE : 1352.48
RMSE: 2395.55
MAPE: 5.14 %


In [7]:
import joblib

joblib.dump(
    model,
    "../models/xgboost_energy_forecaster_v3.pkl"
)

['../models/xgboost_energy_forecaster_v3.pkl']

In [8]:
results = pd.DataFrame({
    "Date": test["SETTLEMENT_DATE"],
    "Actual": y_test,
    "Predicted": predictions
})

results.to_csv(
    "../reports/forecast_results.csv",
    index=False
)

print(results.head())
print(results.shape)

           Date        Actual     Predicted
8370 2024-01-01  25270.187500  26936.802734
8371 2024-01-02  29282.125000  27882.167969
8372 2024-01-03  30733.437500  29529.339844
8373 2024-01-04  32170.625000  31796.583984
8374 2024-01-05  32196.083333  32690.132812
(731, 3)


In [9]:
results["Error"] = (
    results["Actual"]
    - results["Predicted"]
)

results["Absolute_Error"] = (
    abs(results["Error"])
)

results.to_csv(
    "../reports/forecast_results.csv",
    index=False
)

print(results.head())

           Date        Actual     Predicted        Error  Absolute_Error
8370 2024-01-01  25270.187500  26936.802734 -1666.615234     1666.615234
8371 2024-01-02  29282.125000  27882.167969  1399.957031     1399.957031
8372 2024-01-03  30733.437500  29529.339844  1204.097656     1204.097656
8373 2024-01-04  32170.625000  31796.583984   374.041016      374.041016
8374 2024-01-05  32196.083333  32690.132812  -494.049479      494.049479


In [10]:
print(results["Absolute_Error"].describe())

count      731.000000
mean      1352.475440
std       1978.586879
min          1.007161
25%        304.910156
50%        669.587891
75%       1344.734701
max      15280.297526
Name: Absolute_Error, dtype: float64


In [6]:
import pandas as pd

df = pd.read_parquet(
    r"C:\Users\jumma\Downloads\uk-energy-forecasting\data\processed\energy_features_v3.parquet"
)

df.to_csv(
    r"C:\Users\jumma\Downloads\uk-energy-forecasting\data\processed\energy_features_v3.csv",
    index=False
)

print("CSV saved successfully")

CSV saved successfully


In [7]:
import os

print(os.path.exists(
    r"C:\Users\jumma\Downloads\uk-energy-forecasting\data\processed\energy_features_v3.csv"
))

True


In [8]:
print("Peak Demand:", round(df["ND"].max(), 0))
print("Average Demand:", round(df["ND"].mean(), 0))
print("Lowest Demand:", round(df["ND"].min(), 0))

Peak Demand: 50142.0
Average Demand: 33285.0
Lowest Demand: 16487.0


In [9]:
print(df["EMBEDDED_SOLAR_GENERATION"].max())
print(df["EMBEDDED_WIND_GENERATION"].max())
print(df["EMBEDDED_SOLAR_GENERATION"].mean())
print(df["EMBEDDED_WIND_GENERATION"].mean())

4912.0
5708.958333333333
659.1558602853007
938.6577654430395
